In [1]:
import re
import unicodedata
from collections import defaultdict

import pandas as pd

In [ ]:
documents = {
    "D1": "Coastal Café near Malpe Beach serves affordable vegetarian meals.",
    "D2": "Green Bowl in Manipal serves vegetarian and vegan food.",
    "D3": "Beach Shack near Malpe Beach serves seafood and snacks.",
    "D4": "Campus Bistro near MAHE offers affordable coffee and sandwiches.",
    "D5": "Spice Garden in Udupi serves vegetarian meals and seafood.",
    "D6": "Ocean View Café near the beach offers coffee and seafood."
}

for doc_id, text in documents.items():
    print(f"{doc_id} : {text}")

D1 : Coastal Café near Malpe Beach serves affordable vegetarian meals.
D2 : Green Bowl in Manipal serves vegetarian and vegan food.
D3 : Beach Shack near Malpe Beach serves seafood and snacks.
D4 : Campus Bistro near MAHE offers affordable coffee and sandwiches.
D5 : Spice Garden in Udupi serves vegetarian meals and seafood.
D6 : Ocean View Café near the beach offers coffee and seafood.


In [9]:
text = "Coastal Café near Malpe Beach serves affordable vegetarian meals."

tokens = re.findall(r"[\w\d_]+", text, flags=re.UNICODE)
print(tokens)

['Coastal', 'Café', 'near', 'Malpe', 'Beach', 'serves', 'affordable', 'vegetarian', 'meals']


In [ ]:

text = "Coastal Café near Malpe Beach serves affordable vegetarian meals."
tokens = re.search(r"\s+",text,flags=re.UNICODE)
print(tokens.items)

<re.Match object; span=(7, 8), match=' '>


In [13]:
def preprocess(text):
    text=unicodedata.normalize('NFC',text)
    text = text.lower()
    tokens = re.findall(r"[^\W\d_]+",text,flags=re.UNICODE)
    tokens = [token for token in tokens if token!='the']

    return tokens


preprocessed_documents = {
    doc_id : preprocess(text)
    for doc_id,text in documents.items()
}

for doc_id , tokens in preprocessed_documents.items():
    print(doc_id,'->',tokens)

D1 -> ['coastal', 'café', 'near', 'malpe', 'beach', 'serves', 'affordable', 'vegetarian', 'meals']
D2 -> ['green', 'bowl', 'in', 'manipal', 'serves', 'vegetarian', 'and', 'vegan', 'food']
D3 -> ['beach', 'shack', 'near', 'malpe', 'beach', 'serves', 'seafood', 'and', 'snacks']
D4 -> ['campus', 'bistro', 'near', 'mahe', 'offers', 'affordable', 'coffee', 'and', 'sandwiches']
D5 -> ['spice', 'garden', 'in', 'udupi', 'serves', 'vegetarian', 'meals', 'and', 'seafood']
D6 -> ['ocean', 'view', 'café', 'near', 'beach', 'offers', 'coffee', 'and', 'seafood']


In [19]:
term ='vegetarian'
row=[]

for doc_id in documents:
    if term in preprocessed_documents[doc_id]:
        row.append(1)
    else:
        row.append(0)

print("Documents: ", list(documents.keys()))
print(term, ':', row)

Documents:  ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
vegetarian : [1, 1, 0, 0, 1, 0]


In [20]:
term ='coffee'
row=[]

for doc_id in documents:
    if term in preprocessed_documents[doc_id]:
        row.append(1)
    else:
        row.append(0)

print("Documents: ", list(documents.keys()))
print(term, ':', row)

Documents:  ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
coffee : [0, 0, 0, 1, 0, 1]


In [21]:
vocabulary = [
    "affordable",
    "vegetarian",
    "vegan",
    "seafood",
    "beach",
    "coffee"
]

matrix_data = {}

for term in vocabulary:
    row = []

    for doc_id in documents:
        if term in preprocessed_documents[doc_id]:
            row.append(1)
        else:
            row.append(0)

    matrix_data[term] = row

incidence_df = pd.DataFrame(
    matrix_data,
    index=list(documents.keys())
).T

incidence_df

,D1,D2,D3,D4,D5,D6
affordable,1,0,0,1,0,0
vegetarian,1,1,0,0,1,0
vegan,0,1,0,0,0,0
seafood,0,0,1,0,1,1
beach,1,0,1,0,0,1
coffee,0,0,0,1,0,1


In [22]:
print("ROW: vegetarian")
print(incidence_df.loc["vegetarian"])

print("\nCOLUMN: D3")
print(incidence_df["D3"])

ROW: vegetarian
D1    1
D2    1
D3    0
D4    0
D5    1
D6    0
Name: vegetarian, dtype: int64

COLUMN: D3
affordable    0
vegetarian    0
vegan         0
seafood       1
beach         1
coffee        0
Name: D3, dtype: int64


In [24]:
row1=incidence_df.loc['vegetarian']
row2=incidence_df.loc['beach']

and_mask =(row1 == 1) & (row2 == 1)
print(and_mask)
matching_docs = incidence_df.columns[and_mask].tolist()

print('vegetarian and beach -> ',matching_docs)

D1     True
D2    False
D3    False
D4    False
D5    False
D6    False
dtype: bool
vegetarian and beach ->  ['D1']


In [25]:
# OR example: coffee OR vegan
row1 = incidence_df.loc["coffee"]
row2 = incidence_df.loc["vegan"]

or_mask = (row1 == 1) | (row2 == 1)
print("coffee OR vegan ->",
      incidence_df.columns[or_mask].tolist())


# AND NOT example: vegetarian AND NOT seafood
row1 = incidence_df.loc["vegetarian"]
row2 = incidence_df.loc["seafood"]

and_not_mask = (row1 == 1) & (row2 == 0)
print("vegetarian AND NOT seafood ->",
      incidence_df.columns[and_not_mask].tolist())

coffee OR vegan -> ['D2', 'D4', 'D6']
vegetarian AND NOT seafood -> ['D1', 'D2']
